In [2]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\user\Desktop\photo-director-engine\test
['dataprocessing.ipynb', 'dji_bottle.jpeg']


In [3]:
import cv2
import numpy as np
import os
from PIL import Image


def resize_long_side(img, long_side=1024):
    h, w = img.shape[:2]
    scale = long_side / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)


def calc_sharpness(img):
    """
    선명도 지표.`
    값이 클수록 경계가 또렷함.
    """
    img_small = resize_long_side(img, 1024)
    gray = cv2.cvtColor(img_small, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def calc_noise_sigma(img):
    """
    간단한 노이즈 추정.
    값이 클수록 자글자글한 잡음이 많음.
    """
    img_small = resize_long_side(img, 1024)
    gray = cv2.cvtColor(img_small, cv2.COLOR_BGR2GRAY).astype(np.float32)

    smooth = cv2.GaussianBlur(gray, (3, 3), 0)
    residual = gray - smooth

    sigma = np.median(np.abs(residual - np.median(residual))) / 0.6745
    return sigma


def calc_bytes_per_pixel(image_path):
    img = Image.open(image_path)
    w, h = img.size
    file_size = os.path.getsize(image_path)
    return file_size / (w * h)


ref_path = "dji_bottle.jpeg"

ref_img = cv2.imread(ref_path)

target_sharpness = calc_sharpness(ref_img)
target_noise = calc_noise_sigma(ref_img)
target_bpp = calc_bytes_per_pixel(ref_path)

print("목표 선명도:", target_sharpness)
print("목표 노이즈:", target_noise)
print("목표 압축률 bytes/pixel:", target_bpp)

목표 선명도: 292.00286771520047
목표 노이즈: 0.8339511
목표 압축률 bytes/pixel: 0.5406343607401809
